# 00b. Limpeza de dados

Entre o NB00 (duas bases) e a EDA (NB01). Duas classes de problema:

**Linha que não pode casar.** CPF com `ano_obito <= ANO_OBITO_CORTE` sai da base.
`ANO_OBITO_CORTE = 0` desliga o filtro (subir o corte remove *mais*). Sem
`nome_completo` sai nas duas origens.

**Valor-sentinela.** `''` e `'00000000'` viram `NULL`.

Limpa cada tabela: `censo_registros` → `censo_limpo`, `cpf_registros` →
`cpf_limpo`. Carimba `cpf_norm` no Censo com a coorte (`MIN` se ambíguo).
Diagnósticos usam views UNION das duas. O Splink consome as limpas via
`materialize_splink_input`.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import config
from config import (
    CENSO_LIMPO,
    COHORT_DEDUP_ARQUIVO,
    COLUNAS_ESTRUTURAIS,
    CPF_LIMPO,
    TABELA_CENSO_LIMPA,
    TABELA_CENSO_REGISTROS,
    TABELA_CPF_LIMPA,
    TABELA_CPF_REGISTROS,
    sem_nome_sql,
    cpf_norm_sql,
    dob_valida_sql,
    cep_valido_sql,
    export_parquet,
    get_connection,
    limpeza_columns_sql,
    materialize_cohort_cpf_por_censo,
    obito_antes_do_censo_sql,
    print_paths,
    require_input,
    require_tables,
    stamp_censo_cpf_from_cohort,
)

print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
con = get_connection()
require_tables(
    con,
    [TABELA_CENSO_REGISTROS, TABELA_CPF_REGISTROS],
    notebook_origem='00',
)

# Views UNION para diagnóstico (mesmas colunas nas duas bases).
ORIGEM = '_diag_antes'
TABELA_LIMPA = '_diag_depois'
con.execute(f'''
CREATE OR REPLACE VIEW {ORIGEM} AS
SELECT * FROM {TABELA_CENSO_REGISTROS}
UNION ALL
SELECT * FROM {TABELA_CPF_REGISTROS}
''')

tipos = {r[0]: r[1] for r in con.execute(f'DESCRIBE {ORIGEM}').fetchall()}
tipos_censo = {
    r[0]: r[1] for r in con.execute(f'DESCRIBE {TABELA_CENSO_REGISTROS}').fetchall()
}
tipos_cpf = {
    r[0]: r[1] for r in con.execute(f'DESCRIBE {TABELA_CPF_REGISTROS}').fetchall()
}
texto_cols = [
    c for c, t in tipos.items()
    if t.upper().startswith('VARCHAR') and c not in COLUNAS_ESTRUTURAIS
]
n_antes = con.execute(f'SELECT COUNT(*) FROM {ORIGEM}').fetchone()[0]
print(f'{ORIGEM}: {n_antes:,} linhas | {len(texto_cols)} colunas de texto a limpar')
print(f'ANO_OBITO_CORTE={config.ANO_OBITO_CORTE} (0 = não remove ninguém por óbito)')


OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_pessoas_2022_20260505.parquet
CENSO_CEP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/raw/censo/data_cep_uniq.csv
COHORT_DEDUP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/capefe/dados/CohortDados/cohort_dedup.parquet
FILTRO_UF: 21
FILTRO_MUNICIPIO: None
USE_PHONETIC_STRIP_VOWELS: False
ANO_OBITO_CORTE: 2023
ANO_NASCIMENTO_MIN: 1900
SEXO_VALIDOS: ('M', 'F')
DUCKDB_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/probabilistico.duckdb
registro_unificado: 15,222,975 linhas | 21 colunas de texto a limpar


## 1. Diagnóstico antes

Quanto de cada coluna é string vazia hoje. São esses valores que o Splink
compara como se fossem iguais entre si.

In [2]:
vazios = ',\n    '.join(
    f"SUM(CASE WHEN TRIM(CAST({c} AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS {c}"
    for c in texto_cols
)
df_vazios = con.execute(f'''
SELECT origem, COUNT(*) AS n_linhas,
    {vazios}
FROM {ORIGEM} GROUP BY origem ORDER BY origem
''').df().set_index('origem').T
display(df_vazios[df_vazios.sum(axis=1) > 0])

origem,censo,cpf
n_linhas,6775805.0,8447170.0
data_nascimento,998181.0,591.0
cep,486.0,0.0


In [3]:
# Os valores mais frequentes denunciam preenchimento sintético. Se alguma data
# ou CEP aparecer com contagem fora de escala, é sentinela e não dado.
for col in ['data_nascimento', 'cep']:
    print(f'\n=== {col}: 15 valores mais frequentes ===')
    display(con.execute(f'''
    SELECT {col} AS valor, origem, COUNT(*) AS n
    FROM {ORIGEM}
    GROUP BY 1, 2 ORDER BY n DESC LIMIT 15
    ''').df())


=== data_nascimento: 15 valores mais frequentes ===


,valor,origem,n
0,,censo,998181
1,,cpf,591
2,2002-05-26,cpf,550
3,2000-03-20,cpf,536
4,1986-11-15,cpf,535
5,2004-09-09,cpf,529
6,1985-09-07,cpf,519
7,2004-08-26,cpf,508
8,2009-09-09,censo,506
9,2005-05-05,censo,505



=== cep: 15 valores mais frequentes ===


,valor,origem,n
0,65000000,cpf,303462
1,65110000,censo,244343
2,65110000,cpf,181983
3,65400000,cpf,154732
4,65930000,cpf,148946
5,65700000,cpf,134929
6,65130000,cpf,131613
7,65800000,cpf,127268
8,65950000,cpf,117754
9,65130000,censo,116498


In [4]:
# Distribuição completa de sexo: são poucos valores e cabe inteira. A coluna
# 'mantido' mostra o que sobrevive — o resto ('O' de outro, 'I' de ignorado,
# '9' de não informado) vira NULL. Se alguma categoria fora de M/F tiver volume
# relevante e for real, reveja SEXO_VALIDOS em config.py antes de seguir.
sexo_lista = ', '.join(f"'{v}'" for v in config.SEXO_VALIDOS)
display(con.execute(f'''
SELECT
    CASE WHEN TRIM(CAST(sexo AS VARCHAR)) = '' THEN '(vazio)' ELSE sexo END AS valor,
    origem,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY origem), 3) AS pct_origem,
    upper(TRIM(CAST(sexo AS VARCHAR))) IN ({sexo_lista}) AS mantido
FROM {ORIGEM}
GROUP BY 1, 2, 5 ORDER BY origem, n DESC
''').df())

,valor,origem,n,pct_origem,mantido
0,F,censo,3446843,50.870,True
1,M,censo,3328962,49.130,True
2,M,cpf,4289202,50.777,True
3,F,cpf,4156812,49.210,True
4,9,cpf,1156,0.014,False


In [5]:
# Ano de óbito. 'sem óbito' inclui todo o Censo, que nunca tem a coluna.
display(con.execute(f'''
SELECT
    CASE
        WHEN ano_obito IS NULL THEN 'sem ano de óbito'
        WHEN ano_obito <= 0 THEN 'sentinela (<= 0)'
        WHEN ano_obito <= {config.ANO_OBITO_CORTE} THEN 'até o corte (sai)'
        ELSE 'depois do corte (fica)'
    END AS faixa,
    origem,
    COUNT(*) AS n
FROM {ORIGEM}
GROUP BY 1, 2 ORDER BY n DESC
''').df())

n_obito = con.execute(
    f'SELECT COUNT(*) FROM {ORIGEM} WHERE ano_obito IS NOT NULL'
).fetchone()[0]
if n_obito == 0:
    print(
        'ATENÇÃO: nenhum ano de óbito preenchido, e o NB00 falharia se '
        f'{config.CPF_COL_ANO_OBITO} faltasse no bronze. Então o mais provável é '
        'que este registro_unificado seja anterior à coluna: rode o NB00 de novo. '
        'O filtro de óbito abaixo não vai remover nada.'
    )
else:
    display(con.execute(f'''
    SELECT ano_obito, COUNT(*) AS n FROM {ORIGEM}
    WHERE ano_obito IS NOT NULL GROUP BY 1 ORDER BY ano_obito DESC LIMIT 30
    ''').df())


,faixa,origem,n
0,sem ano de óbito,cpf,7980717
1,sem ano de óbito,censo,6775805
2,até o corte (sai),cpf,420597
3,depois do corte (fica),cpf,45856


,ano_obito,n
0,2025,14902
1,2024,30954
2,2023,28676
3,2022,30541
4,2021,34162
5,2020,32749
6,2019,24687
7,2018,21454
8,2017,18878
9,2016,16542


## 2. Aplicar a limpeza

Um passe só: o `WHERE` remove óbitos com `ano_obito <= ANO_OBITO_CORTE` e
registros sem nome (Censo e CPF); a projeção troca os sentinelas por `NULL`. A idade
do CPF acompanha a data de nascimento (anos completos em `DATA_REFERENCIA_IDADE`);
a do Censo vem de `PECP0401` e não é tocada.


In [6]:
def limpar_tabela(origem: str, destino: str, tipos: dict, *, aplica_obito: bool) -> None:
    cols = limpeza_columns_sql(tipos)
    select_sql = ',\n    '.join(
        (alias if expr == alias else f'{expr} AS {alias}')
        for alias, expr in cols.items()
    )
    filtro_obito = obito_antes_do_censo_sql() if aplica_obito else 'FALSE'
    filtro_sem_nome = sem_nome_sql()
    n_antes = con.execute(f'SELECT COUNT(*) FROM {origem}').fetchone()[0]
    n_rem_obito = con.execute(
        f'SELECT COUNT(*) FROM {origem} WHERE {filtro_obito}'
    ).fetchone()[0]
    n_rem_nome = con.execute(f'''
SELECT COUNT(*) FROM {origem}
WHERE NOT {filtro_obito} AND {filtro_sem_nome}
''').fetchone()[0]
    con.execute(f'''
CREATE OR REPLACE TABLE {destino} AS
SELECT
    {select_sql}
FROM {origem}
WHERE NOT {filtro_obito}
  AND NOT {filtro_sem_nome}
''')
    n_depois = con.execute(f'SELECT COUNT(*) FROM {destino}').fetchone()[0]
    print(
        f'{origem} → {destino}: {n_antes:,} → {n_depois:,} '
        f'({n_rem_obito:,} óbito, {n_rem_nome:,} sem nome)'
    )
    colunas = {r[0] for r in con.execute(f'DESCRIBE {destino}').fetchall()}
    faltando = set(tipos) - colunas
    if faltando:
        raise RuntimeError(f'Colunas perdidas em {destino}: {sorted(faltando)}')


print('Filtro de óbito (só CPF):', obito_antes_do_censo_sql())
limpar_tabela(TABELA_CENSO_REGISTROS, TABELA_CENSO_LIMPA, tipos_censo, aplica_obito=False)
limpar_tabela(TABELA_CPF_REGISTROS, TABELA_CPF_LIMPA, tipos_cpf, aplica_obito=True)

con.execute(f'''
CREATE OR REPLACE VIEW {TABELA_LIMPA} AS
SELECT * FROM {TABELA_CENSO_LIMPA}
UNION ALL
SELECT * FROM {TABELA_CPF_LIMPA}
''')

cohort_cpf = materialize_cohort_cpf_por_censo(con, cohort_parquet=COHORT_DEDUP_ARQUIVO)
n_censo_cpf = stamp_censo_cpf_from_cohort(con, table=TABELA_CENSO_LIMPA)
con.execute(f'''
CREATE OR REPLACE VIEW {TABELA_LIMPA} AS
SELECT * FROM {TABELA_CENSO_LIMPA}
UNION ALL
SELECT * FROM {TABELA_CPF_LIMPA}
''')
print(
    f"Censo limpo com CPF da coorte: {n_censo_cpf:,} "
    f"(coorte {cohort_cpf['n_censo_com_cpf_coorte']:,}; "
    f"ambíguos {cohort_cpf['n_censo_cpf_ambiguo']:,})"
)
n_depois = con.execute(f'SELECT COUNT(*) FROM {TABELA_LIMPA}').fetchone()[0]
print(f'UNION limpo: {n_depois:,} linhas')


Filtro de óbito: (TRY_CAST(ano_obito AS INTEGER) IS NOT NULL AND TRY_CAST(ano_obito AS INTEGER) > 0 AND TRY_CAST(ano_obito AS INTEGER) <= 2023)
Filtro sem nome: (NULLIF(TRIM(CAST(nome_completo AS VARCHAR)), '') IS NULL)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15,222,975 → 14,616,377 linhas (420,597 por óbito, 186,001 sem nome)


## 3. Diagnóstico depois

Duas coisas diferentes viram `NULL` e vale separá-las. **Reencodado** é o `''`
que já era ausência e só mudou de grafia. **Descartado** é valor que existia e
foi julgado inválido — data fora da faixa, CEP `00000000`. O segundo número é o
que merece revisão: se estiver alto, a regra pode estar agressiva demais.

In [7]:
import pandas as pd

linhas = []
for col in texto_cols:
    expr = cols[col]
    if expr == col:
        continue
    linhas.append(con.execute(f'''
    SELECT
        '{col}' AS coluna,
        SUM(CASE WHEN TRIM(CAST({col} AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS reencodado,
        SUM(CASE WHEN TRIM(CAST({col} AS VARCHAR)) <> '' AND ({expr}) IS NULL
                 THEN 1 ELSE 0 END) AS descartado
    FROM {ORIGEM}
    WHERE NOT {filtro_obito} AND NOT {filtro_sem_nome}
    ''').df())

resumo = pd.concat(linhas, ignore_index=True)
resumo = resumo[(resumo['reencodado'] > 0) | (resumo['descartado'] > 0)]
display(resumo.sort_values('descartado', ascending=False))


,coluna,reencodado,descartado
16,data_nascimento,971357.0,257541.0
17,sexo,0.0,980.0
18,cep,406.0,0.0


In [8]:
# Amostra do que foi descartado, para conferir se a regra faz sentido.
display(con.execute(f'''
SELECT data_nascimento, COUNT(*) AS n
FROM {ORIGEM}
WHERE TRIM(CAST(data_nascimento AS VARCHAR)) <> ''
  AND ({dob_valida_sql()}) IS NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20
''').df())

display(con.execute(f'''
SELECT cep, COUNT(*) AS n
FROM {ORIGEM}
WHERE TRIM(CAST(cep AS VARCHAR)) <> ''
  AND ({cep_valido_sql()}) IS NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20
''').df())

,data_nascimento,n
0,2023-05-04,337
1,2023-05-18,331
2,2025-04-10,330
3,2023-09-19,329
4,2023-04-25,326
5,2023-03-09,324
6,2025-06-05,323
7,2023-02-27,323
8,2023-01-03,323
9,2023-11-01,323


,cep,n


In [9]:
# Preenchimento por origem depois da limpeza: é o que o Splink vai ver.
cobertura = ',\n    '.join(
    f'ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS {c}' for c in texto_cols
)
display(con.execute(f'''
SELECT origem, COUNT(*) AS n_linhas,
    {cobertura}
FROM {TABELA_LIMPA} GROUP BY origem ORDER BY origem
''').df().set_index('origem').T)

origem,censo,cpf
n_linhas,6589804.0,8026573.0
nome_completo,100.0,100.0
primeiro_nome,100.0,100.0
nome_meio,88.4,96.2
ultimo_nome,99.6,100.0
nome_completo_phon,100.0,100.0
primeiro_nome_phon,100.0,100.0
nome_meio_phon,88.4,96.2
ultimo_nome_phon,99.6,100.0
nome_mae,33.8,96.8


## 4. Impacto na coorte

O filtro de óbito não pode comer ground truth. Se um CPF da coorte tem
`ano_obito <= ANO_OBITO_CORTE` e mesmo assim aparece no Censo 2022, alguma das
duas fontes está errada — e o par sairia da avaliação do 03 sem aviso. A
checagem olha o CPF **e** o `PERSON_ID_CENSO`. A checagem olha o lado
CPF **e** o lado Censo (`PERSON_ID_CENSO`), porque `sem_nome` também remove
registros da ouro.

In [10]:
if not COHORT_DEDUP_ARQUIVO.exists():
    print('Coorte não encontrada, checagem pulada:', COHORT_DEDUP_ARQUIVO)
else:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE _cohort_cpf AS
    SELECT DISTINCT {cpf_norm_sql('CPF_NORM')} AS cpf_norm
    FROM read_parquet('{COHORT_DEDUP_ARQUIVO}')
    WHERE CPF_NORM IS NOT NULL
    ''')

    filtro_obito_b = obito_antes_do_censo_sql('b.ano_obito')
    filtro_sem_nome_b = sem_nome_sql('b.nome_completo')
    display(con.execute(f'''
    WITH na_base AS (
        SELECT u.unique_id, u.ano_obito, u.origem, u.nome_completo
        FROM {ORIGEM} u
        JOIN _cohort_cpf k ON u.cpf_norm = k.cpf_norm
        WHERE u.origem = 'cpf'
    )
    SELECT
        COUNT(*) AS cpf_da_coorte_no_subset,
        SUM(CASE WHEN {filtro_obito_b} THEN 1 ELSE 0 END) AS removidos_por_obito,
        SUM(CASE WHEN NOT {filtro_obito_b} AND {filtro_sem_nome_b}
                 THEN 1 ELSE 0 END) AS removidos_sem_nome,
        SUM(CASE WHEN l.unique_id IS NULL THEN 1 ELSE 0 END) AS removidos_total,
        ROUND(100.0 * SUM(CASE WHEN l.unique_id IS NULL THEN 1 ELSE 0 END)
              / NULLIF(COUNT(*), 0), 3) AS pct
    FROM na_base b
    LEFT JOIN {TABELA_LIMPA} l ON l.unique_id = b.unique_id
    ''').df())

    display(con.execute(f'''
    SELECT u.cpf_norm, u.nome_completo, u.data_nascimento, u.ano_obito
    FROM {ORIGEM} u
    JOIN _cohort_cpf k ON u.cpf_norm = k.cpf_norm
    WHERE u.origem = 'cpf' AND {filtro_obito}
    LIMIT 20
    ''').df())

    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE _cohort_censo AS
    SELECT DISTINCT CAST(PERSON_ID_CENSO AS VARCHAR) AS person_id_censo
    FROM read_parquet('{COHORT_DEDUP_ARQUIVO}')
    WHERE PERSON_ID_CENSO IS NOT NULL
    ''')
    display(con.execute(f'''
    WITH na_base AS (
        SELECT u.unique_id, u.origem, u.nome_completo
        FROM {ORIGEM} u
        JOIN _cohort_censo k ON u.person_id_censo = k.person_id_censo
        WHERE u.origem = 'censo'
    )
    SELECT
        COUNT(*) AS censo_da_coorte_no_subset,
        SUM(CASE WHEN {filtro_sem_nome_b} THEN 1 ELSE 0 END) AS removidos_sem_nome,
        SUM(CASE WHEN l.unique_id IS NULL THEN 1 ELSE 0 END) AS removidos_total,
        ROUND(100.0 * SUM(CASE WHEN l.unique_id IS NULL THEN 1 ELSE 0 END)
              / NULLIF(COUNT(*), 0), 3) AS pct
    FROM na_base b
    LEFT JOIN {TABELA_LIMPA} l ON l.unique_id = b.unique_id
    ''').df())


,cpf_da_coorte_no_subset,removidos_por_obito,removidos_sem_nome,removidos_total,pct
0,1543671,6120.0,0.0,6120.0,0.396


,cpf_norm,nome_completo,data_nascimento,ano_obito
0,11733209204,CILVINO SOUSA SILVA,1956-01-28,2023
1,15829545349,MARIA JOSE LEAO MARQUES,1952-06-01,2023
2,17704960253,LOURIVAL RODRIGUES CRUZ,1960-11-14,2023
3,21610746368,RAIMUNDO JOSE SILVA,1959-06-12,2023
4,00116569301,FRANCINEUDE ALVES SOUSA,1974-12-14,2023
5,00881755370,RUBENS CONCEICAO CHAVIER,1984-08-20,2023
6,01356005373,MARIA GLORIA NASCIMENTO MORAES,1964-01-21,2023
7,01360959335,LUCIANO RODRIGUES QUARESMA SILVA,1996-06-04,2023
8,02964757348,ADAO EVANGELISTA SILVA,1967-05-01,2023
9,03256947379,RAIMUNDA CONRADO ALMEIDA,1951-01-17,2023


## 5. Export

In [11]:
print('Exportado:', export_parquet(con, TABELA_CENSO_LIMPA, path=CENSO_LIMPO))
print('Exportado:', export_parquet(con, TABELA_CPF_LIMPA, path=CPF_LIMPO))
con.close()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exportado: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/registro_limpo.parquet
